In [9]:
# ============================================================
# PHASE 7 — FINAL ML EXPERIMENT
# Historical Churn Classification from customer_360
# ============================================================

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score,
    average_precision_score
)

In [1]:
import os
from dotenv import load_dotenv
from sqlalchemy import create_engine

load_dotenv()

engine = create_engine(os.getenv("DATABASE_URL"))

In [11]:
# ------------------------------------------------------------
# 1. Load customer_360
# ------------------------------------------------------------

df = pd.read_sql(
    "SELECT * FROM customer_360",
    engine
)

print("customer_360:", df.shape)

customer_360: (1200, 30)


In [12]:
# ------------------------------------------------------------
# 2. Create target
# ------------------------------------------------------------

df["churned"] = (
    df["subscription_status"] == "churned"
).astype(int)

print("\nTarget distribution:")
print(df["churned"].value_counts())

print("\nChurn rate:")
print(round(df["churned"].mean() * 100, 2), "%")


Target distribution:
churned
0    1114
1      86
Name: count, dtype: int64

Churn rate:
7.17 %


In [13]:
# ------------------------------------------------------------
# 3. Create useful pre-outcome features
# ------------------------------------------------------------

df["customer_tenure_days"] = (
    pd.to_datetime(df["started_on"])
    - pd.to_datetime(df["signup_date"])
).dt.days

In [14]:
# ------------------------------------------------------------
# 4. Remove explicit leakage / outcome fields
# ------------------------------------------------------------

drop_cols = [
    "account_id",

    # Direct outcome information
    "subscription_status",
    "ended_on",
    "subscription_duration",

    # Raw dates
    "signup_date",
    "started_on",

    # Aggregates that can directly encode the outcome
    # through post-outcome financial activity
    "uncollectible_amount"
]

X = df.drop(
    columns=drop_cols + ["churned"],
    errors="ignore"
)

y = df["churned"]

In [15]:
# ------------------------------------------------------------
# 5. Identify feature types
# ------------------------------------------------------------

categorical_features = X.select_dtypes(
    include=["object", "string"]
).columns.tolist()

numeric_features = X.select_dtypes(
    include=["int64", "float64", "int32", "float32"]
).columns.tolist()

print("\nCategorical features:")
print(categorical_features)

print("\nNumeric features:")
print(numeric_features)


Categorical features:
['company_name', 'industry', 'country', 'plan']

Numeric features:
['employee_count', 'user_count', 'admin_count', 'member_count', 'viewer_count', 'seats', 'mrr', 'users_per_seat', 'invoice_count', 'total_invoiced', 'avg_invoice_amount', 'paid_amount', 'open_amount', 'ticket_count', 'high_priority_tickets', 'urgent_tickets', 'avg_satisfaction', 'avg_resolution_hours', 'unresolved_tickets', 'customer_tenure_days']


In [16]:
# ------------------------------------------------------------
# 6. Preprocessing
# ------------------------------------------------------------

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numeric_features),
    ("categorical", categorical_pipeline, categorical_features)
])

In [17]:
# ------------------------------------------------------------
# 7. Stratified train/test split
# ------------------------------------------------------------

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    stratify=y,
    random_state=42
)

print("\nTrain:", X_train.shape)
print("Test :", X_test.shape)


Train: (960, 24)
Test : (240, 24)


In [18]:
# ------------------------------------------------------------
# 8. Model 1 — Logistic Regression
# ------------------------------------------------------------

logistic_model = Pipeline([
    ("preprocessor", preprocessor),
    ("model", LogisticRegression(
        max_iter=2000,
        class_weight="balanced",
        random_state=42
    ))
])

logistic_model.fit(X_train, y_train)

logistic_prob = logistic_model.predict_proba(X_test)[:, 1]
logistic_pred = logistic_model.predict(X_test)

print("\n==============================")
print("LOGISTIC REGRESSION")
print("==============================")

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, logistic_pred))

print("\nClassification Report:")
print(
    classification_report(
        y_test,
        logistic_pred,
        digits=4
    )
)

print(
    "ROC-AUC:",
    round(roc_auc_score(y_test, logistic_prob), 4)
)

print(
    "PR-AUC:",
    round(average_precision_score(y_test, logistic_prob), 4)
)


LOGISTIC REGRESSION

Confusion Matrix:
[[190  33]
 [ 15   2]]

Classification Report:
              precision    recall  f1-score   support

           0     0.9268    0.8520    0.8879       223
           1     0.0571    0.1176    0.0769        17

    accuracy                         0.8000       240
   macro avg     0.4920    0.4848    0.4824       240
weighted avg     0.8652    0.8000    0.8304       240

ROC-AUC: 0.4012
PR-AUC: 0.0599


In [19]:
# ------------------------------------------------------------
# 9. Model 2 — Random Forest
# ------------------------------------------------------------

rf_model = Pipeline([
    ("preprocessor", preprocessor),
    ("model", RandomForestClassifier(
        n_estimators=500,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1,
        min_samples_leaf=3
    ))
])

rf_model.fit(X_train, y_train)

rf_prob = rf_model.predict_proba(X_test)[:, 1]
rf_pred = rf_model.predict(X_test)

print("\n==============================")
print("RANDOM FOREST")
print("==============================")

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, rf_pred))

print("\nClassification Report:")
print(
    classification_report(
        y_test,
        rf_pred,
        digits=4
    )
)

print(
    "ROC-AUC:",
    round(roc_auc_score(y_test, rf_prob), 4)
)

print(
    "PR-AUC:",
    round(average_precision_score(y_test, rf_prob), 4)
)


RANDOM FOREST

Confusion Matrix:
[[156  67]
 [ 15   2]]

Classification Report:
              precision    recall  f1-score   support

           0     0.9123    0.6996    0.7919       223
           1     0.0290    0.1176    0.0465        17

    accuracy                         0.6583       240
   macro avg     0.4706    0.4086    0.4192       240
weighted avg     0.8497    0.6583    0.7391       240

ROC-AUC: 0.4207
PR-AUC: 0.0672


In [20]:
# ------------------------------------------------------------
# 10. Cross-validation
# ------------------------------------------------------------

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

logistic_cv = cross_val_score(
    logistic_model,
    X,
    y,
    cv=cv,
    scoring="roc_auc",
    n_jobs=-1
)

rf_cv = cross_val_score(
    rf_model,
    X,
    y,
    cv=cv,
    scoring="roc_auc",
    n_jobs=-1
)

print("\n==============================")
print("5-FOLD CROSS-VALIDATION")
print("==============================")

print(
    "Logistic Regression ROC-AUC:",
    round(logistic_cv.mean(), 4),
    "+/-",
    round(logistic_cv.std(), 4)
)

print(
    "Random Forest ROC-AUC:",
    round(rf_cv.mean(), 4),
    "+/-",
    round(rf_cv.std(), 4)
)


5-FOLD CROSS-VALIDATION
Logistic Regression ROC-AUC: 0.5182 +/- 0.0386
Random Forest ROC-AUC: 0.512 +/- 0.0441


In [21]:
# ------------------------------------------------------------
# 11. Feature importance — Random Forest
# ------------------------------------------------------------

fitted_preprocessor = rf_model.named_steps["preprocessor"]
fitted_rf = rf_model.named_steps["model"]

feature_names = fitted_preprocessor.get_feature_names_out()

importance_df = pd.DataFrame({
    "feature": feature_names,
    "importance": fitted_rf.feature_importances_
})

importance_df = importance_df.sort_values(
    "importance",
    ascending=False
)

print("\n==============================")
print("TOP 15 RANDOM FOREST FEATURES")
print("==============================")

print(
    importance_df.head(15).to_string(index=False)
)


TOP 15 RANDOM FOREST FEATURES
                       feature  importance
         numeric__member_count    0.060349
          numeric__paid_amount    0.056596
     numeric__avg_satisfaction    0.053041
   numeric__avg_invoice_amount    0.052480
           numeric__user_count    0.051872
                numeric__seats    0.048060
       numeric__users_per_seat    0.046964
       numeric__employee_count    0.046329
                  numeric__mrr    0.046186
        numeric__invoice_count    0.042554
       numeric__total_invoiced    0.042464
 numeric__avg_resolution_hours    0.038808
         numeric__viewer_count    0.035028
          numeric__open_amount    0.034400
numeric__high_priority_tickets    0.030660


In [22]:
# ------------------------------------------------------------
# 12. Final comparison
# ------------------------------------------------------------

comparison = pd.DataFrame({
    "model": [
        "Logistic Regression",
        "Random Forest"
    ],
    "test_roc_auc": [
        roc_auc_score(y_test, logistic_prob),
        roc_auc_score(y_test, rf_prob)
    ],
    "test_pr_auc": [
        average_precision_score(y_test, logistic_prob),
        average_precision_score(y_test, rf_prob)
    ],
    "cv_roc_auc_mean": [
        logistic_cv.mean(),
        rf_cv.mean()
    ],
    "cv_roc_auc_std": [
        logistic_cv.std(),
        rf_cv.std()
    ]
})

print("\n==============================")
print("FINAL MODEL COMPARISON")
print("==============================")

print(
    comparison.round(4).to_string(index=False)
)

print("\nPhase 7 ML experiment completed.")


FINAL MODEL COMPARISON
              model  test_roc_auc  test_pr_auc  cv_roc_auc_mean  cv_roc_auc_std
Logistic Regression        0.4012       0.0599           0.5182          0.0386
      Random Forest        0.4207       0.0672           0.5120          0.0441

Phase 7 ML experiment completed.
